In [ ]:
aa yahoo 
https://claude.ai/chat/3bd82f2f-db76-4312-ad17-7412b473aae8

In [2]:
# собираем вакансии
import re
import time
from urllib.parse import quote_plus

import pandas as pd
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from webdriver_manager.chrome import ChromeDriverManager

SEARCH_QUERIES = [
    # ── Ввод и обработка данных (лёгкий вход) ────────────
"оператор по вводу данных",
"специалист по вводу данных",
"ввод данных",
"data entry",
"заполнение баз данных",
"наполнение базы данных",

# ── Менеджер с CRM / заказами ────────────────────────
"менеджер по обработке заказов",
"менеджер заказов",
"менеджер CRM",
"оператор CRM",
"менеджер базы данных клиентов",
"специалист CRM",
"администратор CRM",

# ── Документооборот и реестры ────────────────────────
"менеджер по документообороту",
"специалист по документообороту",
"делопроизводитель",
"специалист реестр",
"ведение реестров",

# ── Каталог / контент данных ─────────────────────────
"менеджер каталога",
"специалист по наполнению сайта",
"менеджер товарного каталога",
"контент менеджер каталог",
"менеджер карточек товаров",
"специалист по карточкам товаров",

# ── Координатор / операционист ───────────────────────
"операционный менеджер",
"координатор данных",
"менеджер по данным",
    # ── Ядро ─────────────────────────────────────
    "аналитик данных",
    "data analyst",
    "дата аналитик",
    "аналитик по данным",
    "специалист по данным",
    "специалист данных",

    # ── BI и визуализация ────────────────────────
    "BI аналитик",
    "аналитик BI",
    "BI разработчик",
    "Power BI аналитик",
    "Tableau аналитик",
    "DataLens аналитик",
    "дашборд специалист",
    "аналитик дашбордов",

    # ── Продуктовая / маркетинговая ──────────────
    "продуктовый аналитик",
    "product analyst",
    "маркетинговый аналитик",
    "growth analyst",
    "веб аналитик",
    "web analyst",

    # ── Финансовая аналитика с данными ───────────
    "финансовый аналитик Excel",
    "финансовый аналитик SQL",
    "финансовый аналитик данные",
    "финансовый аналитик Python",
    "финансовый моделист",
    "финансовое моделирование",

    # ── Отчётность и расчёты ─────────────────────
    "аналитик отчётности",
    "аналитик отчетности",
    "специалист отчётности",
    "специалист по отчётам",
    "расчётчик",
    "расчетчик",
    "оператор данных",

    # ── Инструменты как название должности ───────
    "SQL аналитик",
    "Python аналитик",
    "Excel аналитик",
    "Google Sheets аналитик",

    # ── Качество и обработка данных ──────────────
    "data quality",
    "аналитик качества данных",
    "data steward",
    "обработка данных",
    "сбор данных",
]

BASE_URL = (
    "https://hh.ru/search/vacancy"
    "?ored_clusters=true"
    "&area=194"
    "&area=113"
    "&area=1"
    "&area=68"
    "&area=40"
    "&suggestId=dddc5521-9655-4d70-8087-b312596ea2cb"
    "&hhtmFrom=vacancy_search_list"
    "&work_format=REMOTE"
)

MAX_PAGES = 5
PAUSE     = 2.0
HEADLESS  = False

# ══════════════════════════════════════════════════════
#  🔍 Паттерны для разбора текстов карточки
# ══════════════════════════════════════════════════════

NOISE_WORDS = {
    "откликнуться", "контакты", "интервью о жизни в компании",
    "можно удалённо", "отзывов", "спортивная",
}

RE_NOT_CITY = re.compile(
    r"(₸|₽|\$|€|выплат|месяц|человек|сейчас|откликнуться|контакты|"
    r"интервью|рейтинг|тоо|оо |ао |ао$|чк |ип |зао|пао|"
    r"bank|group|tech|digital|finance|insurance|center|ltd|llc|inc|"
    r"компани|банк|школ|групп|центр|академи|страхов|институт|"
    r"отзыв|\d{4,})",
    re.IGNORECASE,
)

# ИСПРАВЛЕНО: убрали обязательное слово «месяц» — теперь ловим любую сумму с валютой
RE_SALARY = re.compile(
    r"(₸|₽|\$|€).{0,80}?(\d[\d\s]*\d|\d+)"   # валюта справа от суммы
    r"|(\d[\d\s]*\d|\d+).{0,20}?(₸|₽|\$|€)",   # или валюта слева
    re.IGNORECASE,
)

RE_EXPERIENCE = re.compile(r"(без опыта|опыт\s[\d\w].{0,30})", re.IGNORECASE)

KNOWN_CITIES = {
    "алматы", "астана", "нур-султан", "шымкент", "москва",
    "санкт-петербург", "екатеринбург", "новосибирск", "омск",
    "минск", "ташкент", "бишкек", "тбилиси", "ереван", "кейптаун",
    "алма-ата", "актау", "актобе", "атырау", "костанай", "павлодар",
    "семей", "уральск", "усть-каменогорск", "тараз", "петропавловск",
    "краснодар", "казань", "нижний новгород", "воронеж", "самара",
    "ростов-на-дону", "уфа", "пермь", "волгоград", "красноярск",
    "россия", "казахстан",
    "кызылорда", "туркестан", "кокшетау", "талдыкорган", "жезказган",
    "темиртау", "балхаш", "рудный", "экибастуз", "степногорск",
    "кентау", "жанаозен", "аксай", "аркалык", "лисаковск",
    "риддер", "шахтинск", "сатпаев", "байконур", "капшагай",
    "щучинск", "талгар", "каскелен", "конаев", "текели",
    "хромтау", "жаркент", "аягоз", "алтай", "шемонаиха",
    "сергеевка", "зайсан", "курчатов", "жетысай", "сарыагаш",
    "арысь", "отырар", "форт-шевченко", "ленгер", "каратау",
    "жанатас", "шу", "отеген батыр",
    "челябинск", "тюмень", "иркутск", "хабаровск", "владивосток",
    "барнаул", "новокузнецк", "кемерово", "томск", "оренбург",
    "саратов", "тольятти", "ульяновск", "набережные челны",
    "ижевск", "ярославль", "астрахань", "рязань", "пенза",
    "липецк", "киров", "чебоксары", "курск", "тула",
    "брянск", "магнитогорск", "нижний тагил", "сочи", "ставрополь",
    "белгород", "владимир", "симферополь", "севастополь", "иваново",
    "владикавказ", "якутск", "улан-удэ", "чита", "калининград",
    "смоленск", "орел", "тверь", "мурманск", "архангельск",
    "вологда", "псков", "великий новгород", "петрозаводск",
    "сыктывкар", "нальчик", "грозный", "магас", "майкоп",
    "черкесск", "махачкала", "биробиджан", "южно-сахалинск",
    "петропавловск-камчатский", "магадан", "анадырь", "салехард",
    "ханты-мансийск", "сургут", "нижневартовск", "тобольск",
    "нефтеюганск", "ноябрьск", "нижнекамск", "альметьевск",
    "стерлитамак", "таганрог", "шахты", "новороссийск", "армавир",
    "балашиха", "подольск", "химки", "электросталь", "мытищи",
    "люберцы", "королев", "коломна", "серпухов", "орехово-зуево",
    "дзержинск", "арзамас", "обнинск", "калуга", "кострома",
    "дербент", "каспийск", "хасавюрт", "буйнакск", "кизляр",
    "элиста", "абакан", "горно-алтайск", "кызыл", "нерюнгри",
    "братск", "ангарск", "усть-илимск", "норильск", "дудинка",
    "воркута", "ухта", "усинск", "инта", "нарьян-мар",
    "новый уренгой", "надым", "когалым", "лангепас", "мегион",
    "белово", "ленинск-кузнецкий", "прокопьевск", "киселевск",
    "бийск", "рубцовск", "заринск", "искитим", "бердск",
    "озерск", "снежинск", "трехгорный", "аша", "миасс",
    "златоуст", "копейск", "троицк", "чесма", "лысьва",
    "березники", "соликамск", "чусовой", "кунгур", "краснокамск",
    "сарапул", "воткинск", "глазов", "можга", "камышин",
    "михайловка", "волжский", "балаково", "энгельс", "вольск",
    "новочеркасск", "азов", "батайск", "волгодонск", "сальск",
    "геленджик", "анапа", "туапсе", "темрюк", "тихорецк",
    "абинск", "кропоткин", "тимашевск", "ейск", "курганинск",
    "пятигорск", "кисловодск", "ессентуки", "минеральные воды",
    "невинномысск", "буденновск", "георгиевск", "лермонтов",
    "прохладный", "баксан", "нарткала", "тырныауз", "беслан",
    "моздок", "новоалтайск", "камень-на-оби", "ачинск", "канск",
    "минусинск", "лесосибирск", "зеленогорск", "железногорск",
    "боровичи", "старая русса", "великие луки", "невель",
    "гатчина", "выборг", "кронштадт", "тихвин", "луга",
    "кировск", "апатиты", "кандалакша", "мончегорск", "североморск",
    "костомукша", "сортавала", "кемь", "беломорск", "питкяранта",
    "йошкар-ола", "саранск", "чайковский", "губаха",
    "орск", "новотроицк", "бузулук", "бугуруслан", "ясный",
    "балей", "борзя", "краснокаменск", "петровск-забайкальский",
    "свободный", "белогорск", "благовещенск", "тында", "зея",
    "николаевск-на-амуре", "комсомольск-на-амуре", "советская гавань",
    "амурск", "бикин", "вяземский", "уссурийск", "находка",
    "арсеньев", "дальнереченск", "партизанск", "спасск-дальний",
    "фокино", "большой камень",
}


def get_total_pages(driver) -> int:
    """
    Читает пагинатор hh.ru и возвращает количество страниц.
    Ищем все ссылки [data-qa='pager-page'] и берём максимальный номер.
    Если пагинатора нет (результатов меньше одной страницы) — возвращаем 1.
    """
    try:
        # Ждём пока пагинатор появится (может не быть если страница одна)
        pager_links = driver.find_elements(By.CSS_SELECTOR, "[data-qa='pager-page']")
        if not pager_links:
            return 1
        pages = []
        for link in pager_links:
            txt = link.text.strip()
            if txt.isdigit():
                pages.append(int(txt))
        return max(pages) if pages else 1
    except Exception:
        return 1


def parse_texts(texts: list) -> dict:
    result = {"зарплата": "не указана", "опыт": "не указан", "город": "не указан"}

    for i, t in enumerate(texts):
        t = t.strip()
        t_lower = t.lower()

        # ── Зарплата ──────────────────────────────────
        # ИСПРАВЛЕНО: теперь ловим зарплаты и без слова «месяц»
        if result["зарплата"] == "не указана" and RE_SALARY.search(t):
            result["зарплата"] = t

        # ── Опыт ──────────────────────────────────────
        if result["опыт"] == "не указан":
            m = RE_EXPERIENCE.search(t)
            if m:
                result["опыт"] = m.group(0).strip()

        # ── Город (начиная с позиции 2) ───────────────
        if result["город"] == "не указан" and i >= 2:
            city_candidate = t.split(",")[0].strip()
            city_candidate = re.sub(r"\s*\(.*?\)", "", city_candidate).strip()

            if city_candidate.lower() in KNOWN_CITIES:
                result["город"] = city_candidate
                continue

            if (
                not RE_NOT_CITY.search(t)
                and t_lower not in NOISE_WORDS
                and not any(n in t_lower for n in NOISE_WORDS)
                and re.match(r"^[а-яёА-ЯЁa-zA-Z][а-яёА-ЯЁa-zA-Z\s\-\.]*$", city_candidate)
                and 3 <= len(city_candidate) <= 40
            ):
                result["город"] = city_candidate

    return result


def make_driver() -> webdriver.Chrome:
    opts = webdriver.ChromeOptions()
    if HEADLESS:
        opts.add_argument("--headless=new")
    opts.add_argument("--no-sandbox")
    opts.add_argument("--disable-dev-shm-usage")
    opts.add_argument("--window-size=1920,1080")
    opts.add_argument(
        "user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
        "AppleWebKit/537.36 (KHTML, like Gecko) "
        "Chrome/122.0.0.0 Safari/537.36"
    )
    return webdriver.Chrome(
        service=Service(ChromeDriverManager().install()), options=opts
    )


def build_url(query: str, page: int) -> str:
    return f"{BASE_URL}&text={quote_plus(query)}&page={page}"


def get_all_texts(card) -> list:
    texts = []
    seen  = set()

    for el in card.find_elements(By.XPATH, ".//*"):
        try:
            raw = el.parent.execute_script(
                """
                var el = arguments[0], text = '';
                for (var i = 0; i < el.childNodes.length; i++)
                    if (el.childNodes[i].nodeType === 3)
                        text += el.childNodes[i].textContent;
                return text.trim();
                """,
                el,
            )
            if raw and len(raw) > 1 and raw not in seen:
                seen.add(raw)
                texts.append(raw)
        except Exception:
            continue

    return texts


def get_title_and_link(card) -> tuple:
    try:
        a = card.find_element(By.CSS_SELECTOR, "[data-qa='serp-item__title']")
        return a.text.strip(), a.get_attribute("href") or ""
    except Exception:
        return "", ""


def get_company(card) -> str:
    try:
        return card.find_element(
            By.CSS_SELECTOR, "[data-qa='vacancy-serp__vacancy-employer-text']"
        ).text.strip()
    except Exception:
        return "—"


def parse_page(driver) -> list:
    vacancies = []

    try:
        WebDriverWait(driver, 12).until(
            EC.presence_of_all_elements_located(
                (By.CSS_SELECTOR, "[data-qa='vacancy-serp__vacancy']")
            )
        )
    except Exception:
        print("    ⚠️  Карточки не найдены")
        return vacancies

    cards = driver.find_elements(By.CSS_SELECTOR, "[data-qa='vacancy-serp__vacancy']")
    print(f"    📦 Карточек: {len(cards)}")

    for card in cards:
        title, link = get_title_and_link(card)
        if not title:
            continue

        all_texts = get_all_texts(card)
        parsed    = parse_texts(all_texts)

        all_texts_str = " | ".join(f"[{i}]{t}" for i, t in enumerate(all_texts))

        vacancies.append({
            "название":   title,
            "компания":   get_company(card),
            "зарплата":   parsed["зарплата"],
            "опыт":       parsed["опыт"],
            "город":      parsed["город"],
            "ссылка":     link,
            "все_тексты": all_texts_str,
        })

    return vacancies
def scrape() -> pd.DataFrame:
    all_data = []
    seen     = set()

    driver = make_driver()  # ← один раз до цикла

    try:
        for i, query in enumerate(SEARCH_QUERIES):
            print(f"\n{'═'*55}")
            print(f"🔍 [{i+1}/{len(SEARCH_QUERIES)}] «{query}»")
            print(f"{'═'*55}")

            query_count = 0

            try:
                driver.get(build_url(query, 0))
                time.sleep(PAUSE)

                total_pages = get_total_pages(driver)
                total_pages = min(total_pages, MAX_PAGES)
                print(f"  📑 Страниц найдено: {total_pages}")

                for page in range(total_pages):
                    print(f"\n  📄 Страница {page + 1} / {total_pages}")

                    if page > 0:
                        try:
                            driver.get(build_url(query, page))
                            time.sleep(PAUSE)
                        except Exception as e:
                            print(f"  ⚠️  Ошибка загрузки страницы {page}: {e}")
                            break

                    page_data = parse_page(driver)

                    if not page_data:
                        print("  ⏹  Страница пустая — останавливаемся")
                        break

                    new = 0
                    for v in page_data:
                        lnk = v.get("ссылка", "")
                        if lnk not in seen:
                            seen.add(lnk)
                            v["запрос"] = query
                            all_data.append(v)
                            new += 1

                    query_count += new
                    print(f"  ✅ Новых: {new} | По запросу: {query_count} | Всего: {len(all_data)}")

            except Exception as e:
                print(f"  ❌ Критическая ошибка на запросе «{query}»: {e}")
                print(f"  ⏭  Пропускаем и идём дальше")

    finally:
        driver.quit()  # ← закрываем один раз после всех запросов

    cols = ["дата", "название", "зарплата", "опыт", "город",
            "ссылка", "компания", "запрос", "все_тексты"]
    df = pd.DataFrame(all_data)
    if not df.empty:
        df['дата'] = pd.Timestamp.today().date()
        df = df[cols]
    return df

def parse_salary(value):
    if pd.isna(value):
        return None

    text = str(value).strip()

    if re.fullmatch(r'не указана', text.strip(), re.IGNORECASE):
        return None

    cleaned = re.sub(r'[^\d\s\-–]', ' ', text)
    numbers = re.findall(r'\d[\d\s]*\d|\d+', cleaned)

    parsed = []
    for n in numbers:
        num_str = re.sub(r'\s+', '', n)
        if num_str.isdigit():
            parsed.append(int(num_str))

    if not parsed:
        return None

    return min(parsed)


def extract_experience(value):
    match = re.search(r'\d+-\d+', str(value))
    if match:
        return match.group()
    return 'не требуется'


def extract_city_from_all_texts(all_texts_value: str, city_list: list) -> str:
    """
    Ищет город из списка city_list в строке все_тексты.
    ИСПРАВЛЕНО: используем границы слова \b чтобы «Омск» не находился в «объём»
    """
    if pd.isna(all_texts_value):
        return 'не указан'

    text = str(all_texts_value).lower()

    for city in city_list:
        # ИСПРАВЛЕНО: \b границы слова вместо простого "in"
        pattern = r'\b' + re.escape(city.lower()) + r'\b'
        if re.search(pattern, text):
            return city

    return 'не указан'


from openpyxl.styles import Font
import os
def save_excel(df: pd.DataFrame, filename: str = "hh_vacancies.xlsx"):
    if os.path.exists(filename):
        df_existing = pd.read_excel(filename, sheet_name="Вакансии")
        print(f"📂 Существующих строк: {len(df_existing)}, новых: {len(df)}")
    else:
        df_existing = pd.DataFrame()
        print(f"📂 Файл не найден — создаём новый")

    if "статус" not in df.columns:
        df.insert(0, "статус", "")
    if not df_existing.empty and "статус" not in df_existing.columns:
        df_existing["статус"] = ""

    KEY = ["название", "компания", "зарплата", "опыт"]

    if not df_existing.empty:
        KEY = ["название", "компания", "зарплата", "опыт"]
        
        def make_key(row):
            return tuple(str(row[k]).strip() if pd.notna(row[k]) else "" for k in KEY)
    
        status_map = {}
        for _, row in df_existing.drop_duplicates(subset=KEY, keep="first").iterrows():
            status_map[make_key(row)] = row["статус"]
    
        def assign_status(row):
            key = make_key(row)
            existing_status = status_map.get(key, None)
            if existing_status is not None and str(existing_status).strip() not in ("", "nan"):
                return existing_status
            elif key in status_map:
                return "Повтор"
            return ""
    
        df["статус"] = df.apply(assign_status, axis=1)
        df_combined = pd.concat([df_existing, df], ignore_index=True)
    
    else:
        df_combined = df.copy()  # ← этот блок оставляем как есть

    # Удаляем дубликаты — оставляем ПОСЛЕДНЮЮ (новую строку, уже с правильным статусом)
    df_combined = df_combined.drop_duplicates(subset=KEY, keep="last")
    df_combined = df_combined.reset_index(drop=True)

    # Гарантируем порядок колонок
    if "статус" not in df_combined.columns:
        df_combined.insert(0, "статус", "")
    cols = ["статус"] + [c for c in df_combined.columns if c != "статус"]
    df_combined = df_combined[cols]

    print(f"✅ Итого строк после дедупликации: {len(df_combined)}")

    # === Запись в Excel (эта часть не меняется) ===
    url_columns = [
        col for col in df_combined.columns
        if df_combined[col].dtype == object
        and df_combined[col].dropna().astype(str).str.startswith("http").any()
    ]

    with pd.ExcelWriter(filename, engine="openpyxl") as writer:
        df_combined.to_excel(writer, index=False, sheet_name="Вакансии")
        from openpyxl.worksheet.datavalidation import DataValidation

        dv = DataValidation(
            type="list",
            formula1='"Мусор,Следующее,Требования,Сложно,Отправлено,Next!!!,Повтор,Stop,Send-hh"',
            allow_blank=True,
            showDropDown=False,
        )
        ws = writer.sheets["Вакансии"]
        ws.add_data_validation(dv)
        dv.sqref = f"A2:A{len(df_combined) + 1}"

        link_font = Font(color="0563C1", underline="single")
        for col_name in url_columns:
            col_idx = df_combined.columns.get_loc(col_name) + 1
            for row_idx, value in enumerate(df_combined[col_name], start=2):
                if pd.notna(value) and str(value).startswith("http"):
                    cell = ws.cell(row=row_idx, column=col_idx)
                    cell.hyperlink = str(value)
                    cell.font = link_font

        if "дата" in df_combined.columns:
            date_col_idx = df_combined.columns.get_loc("дата") + 1
            for row_idx in range(2, len(df_combined) + 2):
                cell = ws.cell(row=row_idx, column=date_col_idx)
                cell.number_format = "DD.MM.YYYY"

        for col in ws.columns:
            max_len = max(
                (len(str(cell.value)) for cell in col if cell.value), default=10
            )
            ws.column_dimensions[col[0].column_letter].width = min(max_len + 4, 80)

    print(f"💾 Сохранено в: {filename}")
ALLOW = [
    # ── Ввод данных ──────────────────────────────────────
r'\bввод\s*данных\b',
r'\bdata\s*entry\b',
r'\bоператор\s+по\s+вводу\s+данных\b',
r'\bспециалист\s+по\s+вводу\s+данных\b',
r'\bнаполнени.{0,10}баз.{0,5}данн',
r'\bзаполнени.{0,10}баз.{0,5}данн',

# ── CRM-менеджер ─────────────────────────────────────
r'\b(менеджер|специалист|администратор|оператор)\s+(по\s+)?crm\b',
r'\bcrm\s*(менеджер|специалист|оператор|администратор)\b',
r'\bменеджер.{0,20}(клиентск.{0,10}баз|база.{0,5}клиент)',

# ── Заказы ───────────────────────────────────────────
r'\bменеджер\s+по\s+обработке\s+заказов\b',
r'\bменеджер\s+заказов\b',
r'\bоператор\s+заказов\b',

# ── Каталог / карточки ───────────────────────────────
r'\bменеджер\s+(товарного\s+)?каталога\b',
r'\bменеджер\s+карточек?\s+товаров?\b',
r'\bспециалист\s+по\s+карточкам\b',
r'\bнаполнени.{0,10}(каталог|сайт)',

# ── Документооборот / реестры ────────────────────────
r'\bделопроизводитель\b',
r'\bспециалист\s+по\s+документооборот',
r'\bменеджер\s+по\s+документооборот',
r'\bведени.{0,10}реестр',
    # ── Классическая аналитика данных ────────────
    r'\bdata\s*(analyst|engineer|scientist|quality)\b',
    r'\bаналитик\s+данных\b',
    r'\bаналитик\s+по\s+данных\b',
    r'\bдата[-\s]?аналитик\b',
    r'\bпродуктовый\s*аналитик\b',
    r'\bproduct\s*anal[iy]st\b',
    r'\bgrowth\s*anal[iy]st\b',

    # ── BI и визуализация ────────────────────────
    r'\bbi[-\s]?аналитик\b|\bаналитик\s+bi\b',
    r'\bби[-\s]?аналитик\b',
    r'\b(дашборд|dashboard).{0,20}(специалист|разработч|аналитик)',
    r'\bвизуализ.{0,20}данн',
    r'\btableau\b|\bpower\s*bi\s*(аналитик|специалист|разработч)',
    r'\bqlik\b|\blooker\b|\bmetabase\b|\bdatalens\b',

    # ── Маркетинговая и веб-аналитика ────────────
    r'\bмаркетинговый\s*аналитик\b',
    r'\bвеб[-\s]?аналитик\b|\bweb[-\s]?anal[iy]st\b',
    r'\bмедиа.{0,5}аналитик\b',
    r'\bcontentный\s*аналитик\b|\bконтентный\s*аналитик\b',

    # ── Финансовая аналитика с данными ───────────
    r'\bфинансовый\s*аналитик\b',
    r'\bинвестиционный\s*аналитик\b',
    r'\bкредитный\s*аналитик\b',
    r'\bфинансов.{0,20}(модел|расчёт|расчет|планирован).{0,20}(excel|python|sql|google)',
    r'\b(финансов|экономическ).{0,10}модел(ирован|ист)',
    r'\bфинансов.{0,10}(планировщик|прогнозист)',

    # ── Расчёты — НОВОЕ ──────────────────────────
    r'\bрасч[её]тчик\b',
    r'\bспециалист.{0,30}расч[её]т',

    # ── Специалист / работа с данными ────────────
    r'\bспециалист\s+по\s+данным\b',
    r'\bспециалист\s+данных\b',
    r'\bработ.{0,10}(с данными|данных).{0,20}(sql|python|excel|bi\b)',
    r'\bобработк.{0,10}данн',
    r'\bинженер\s+данных\b|\bдата\s+инженер\b',
    r'\bспециалист.{0,20}(данн|data).{0,30}(sql|python|excel|etl|dwh)',

    # ── Отчётность ───────────────────────────────
    r'\bаналитик\s+отч[её]тности\b',
    r'\bспециалист\s+по\s+отч[её]там\b',
    r'\bспециалист\s+отч[её]тности\b',
    r'\b(bi|data|управленческ).{0,30}отч[её]тност',
    r'\bотч[её]тност.{0,30}(bi\b|data|управленческ|msfo|мсфо|dwh)',

    # ── Качество данных — НОВОЕ ──────────────────
    r'\bкачество\s*данных\b|\bкачеству\s*данных\b',
    r'\bdata\s*quality\b|\bdata\s*steward\b',
    r'\bquality\s*anal[iy]st\b',
    r'\bаналитик.{0,20}качеств',
    r'\bсбор\s*данных\b|\bсборщик\s*данных\b',
    r'\bобработка\s*данных\b|\bработа\s*с\s*данными\b',
    r'\bоператор\s+данных\b',

    # ── SQL / Excel / инструменты ────────────────
    r'\b(sql|dwh|etl|llm|crm)\s*(аналитик|analyst)\b',
    r'\bexcel.{0,20}(специалист|оператор|аналитик)',
    r'\bgoogle\s*sheets.{0,20}специалист',
    r'\b(биомедицинск|математическ).{0,10}статистик',
    r'\bстатистик.{0,20}(данн|исследован|анализ)',

    # ── Прочее ───────────────────────────────────
    r'\bаналитик\b.{0,30}\bданн(ых|ыми)\b',
    r'\bаналитик\b.{0,30}\b(визуализ|отч[её]т|статистик|excel|google\s*sheets)\b',
    r'\bаналитик\b.{0,30}\bbi\b',
    r'\b(специалист|эксперт).{0,20}(данн|data|визуализ|отч[её]тност|bi\b)',
    r'\bспециалист.{0,30}\bрасч[её]т',
    r'\banalytics\s+engineer\b',
    r'\bfullstack.{0,5}аналитик\b',
    r'\bantifr[ao]ud.{0,5}(аналитик|analyst)\b',
    r'\bаналитик\s+(маркетплейс|e-?commerce|cltv|роста|трафика)\b',
    r'\bсоциолог.{0,5}аналитик\b',
    r'\banalyst\b',
    r'^аналитик$',
]

DENY = [
    r'\bначальник\b|\bруководитель\b',
r'\bменеджер\s+по\s+продажам\b',
r'\bменеджер\s+по\s+работе\s+с\s+клиентами\b(?!.{0,20}crm)',
r'\bменеджер\s+проект',
r'crm.{0,20}(разработч|developer|программист)',
    # ── Уровень: не Middle ───────────────────────
    r'\bsenior\b',
    r'\blead\b|\bлид\b',
    r'\bteam\s*lead\b',
    r'\bстарший\b',
    r'\bведущий\b',
    r'\bглавный\b|\bглавный\s*специалист\b',
     r'\bпрактикант\b|\btrainee\b|\bстажёр\b|\bстажер\b',

    # ── Бизнес- и системный аналитик ─────────────
    r'\bбизнес[-\s]?аналитик\b',
    r'\bbusiness[-\s]?anal[iy]st\b',
    r'\bсистемный\s*аналитик\b|\bsystem\s*anal[iy]st\b',
    r'system\s*anal[iy]st\s*dwh\b',

    # ── Data Engineer / Scientist / ML ───────────
    r'\bdata\s*engineer\b',
    r'\bинженер\s*данных\b|\bдата\s*инженер\b',
    r'\blead\s*data\s*engineer\b',
    r'\bdata\s*scientist\b',
    r'\bml\s*engineer\b',
    r'\bapplied\s*scientist\b',
    r'\bnlp\b|\bllm\s*(engineer|разработч)\b',
    r'\brecsys\b',

    # ── BI-разработчик (не аналитик) ─────────────
    r'\bbi[-\s]?(developer|разработч|integrat|интегратор)\b',
    r'\bpaginated\s*reports\b',

    # ── Игры / GameDev ───────────────────────────
    r'\bgame\s*anal[iy]st\b|\bгейм\s*аналитик\b',
    r'\bgamedev\b',
    r'\bигровой\s*дизайнер\b|\bgame\s*designer\b',

    # ── Product Manager / другое управление ──────
    r'\bproduct\s*manager\b',
    r'\bдиректор\b|\bcfo\b|\bcpo\b|\bcoo\b|\bcmo\b|\bceo\b|\bcbdo\b',
    r'\bemployer\s*branding\b',

    # ── Расчётчик НЕ данных ──────────────────────
    r'расч[её]т.{0,20}(зарплат|оклад|тариф|опалубк|трубо|инсоляц|тепл|нвф|кэо\b|тэп\b)',
    r'(зарплат|оклад).{0,20}расч[её]т',
    r'\bсметчик\b|\bсмет.{0,5}расч[её]т',
    r'расч[её]т.{0,10}(страхов|осаго|каско)',
    r'расч[её]т.{0,20}(самозанят|зарплат|оклад|жби|ремонт|стоимост)',
    r'\bсамозанят\b',
    r'инженер.{0,20}(расчет|трубо|тепл|электр|гальван|фасад|опалуб)',

    # ── Оператор НЕ данных ───────────────────────
    r'оператор\s*(call|колл|контакт|пк\b|кассы?|1[сc])',

    # ── Отчётность НЕ данных ─────────────────────
    r'отч[её]тност.{0,20}(таможен|под\/фт|фрому|кадров|персонал)',
    r'(кадров|персонал).{0,20}отч[её]тност',

    # ── Финансы без аналитики ────────────────────
    r'\bбухгалтер\b|\bглавбух\b',
    r'\bказначей\b|\bдебиторск\b',
    r'\b1[сc]\b',
    r'\bзаработной[-\s]платы\b',
    r'\bэкономист\b(?!.{0,5}аналитик)',

    # ── Прочий мусор ─────────────────────────────
    r'шофер|водитель|курьер|грузчик|кладовщик',
    r'\bблог(ер|гер)\b|\bамбассадор\b|\bsmm\b|\bugc\b',
    r'\bрепетитор\b|\bпреподаватель\b|\bментор\b',
    r'\bпомощник\b|\bассистент\b',
    r'\bрекрутер\b|\bhr\b',
    r'\bпродавец\b|\bброкер\b|\bриелтор\b|\bпредставитель\b',
    r'\bпентест|\bзащищённост|\bбезопасност',
    r'\bдекларант\b|\bтаможен\b',
    r'\btraffi[ck]\s*analyst\b',
    r'\bquantitative\s*researcher\b',
    r'\bавитолог\b|\bдиректолог\b|\bтаргетолог\b',
    r'(personal|личный)\s*(помощник|ассист)',
    r'\bлейк\b|\blake\s*data\b|\bозеро\s*данных\b',
    r'\bриск\s*аналитик\b|\brisk\s*anal[iy]st\b',
    r'\bбонус.{0,5}аналитик\b|\bbonus\s*anal[iy]st\b',
    r'\bsap\s*hana\b|\boracle\b',
    r'\bc&b\b|\bcomp.{0,5}benefits\b',
    r'\bбинарн.{0,10}опцион\b',
]


def is_target_vacancy(title: str) -> bool:
    t = title.strip()
    allow_hit = any(re.search(p, t, re.IGNORECASE) for p in ALLOW)
    if not allow_hit:
        return False
    deny_hit = any(re.search(p, t, re.IGNORECASE) for p in DENY)
    return not deny_hit


if __name__ == "__main__":
    df = scrape()

    df['зарплата'] = df['зарплата'].apply(parse_salary)
    df['опыт']     = df['опыт'].apply(extract_experience)
    # ИСПРАВЛЕНО: передаём KNOWN_CITIES (список) вместо несуществующей CITIES
    df['город']    = df['все_тексты'].apply(
        lambda x: extract_city_from_all_texts(x, list(KNOWN_CITIES))
    )
    df.drop_duplicates(
        subset=['название', 'компания', 'зарплата', 'опыт', 'город'],
        keep='first', inplace=True
    )
    df.reset_index(drop=True, inplace=True)

    print(f"Строк до фильтрации: {len(df)}\n")

    mask        = df['название'].apply(is_target_vacancy)
    df_filtered = df[mask].reset_index(drop=True)
    removed     = df[~mask]

    print("=== Удалённые строки ===")
    print(f"Удалено: {len(removed)}\n")

    print("=== ПОСЛЕ фильтрации ===")
    print(f"Строк: {len(df_filtered)}")

    save_excel(df_filtered)


═══════════════════════════════════════════════════════
🔍 [1/69] «оператор по вводу данных»
═══════════════════════════════════════════════════════
  📑 Страниц найдено: 5

  📄 Страница 1 / 5
    📦 Карточек: 49
  ✅ Новых: 49 | По запросу: 49 | Всего: 49

  📄 Страница 2 / 5
    📦 Карточек: 50
  ✅ Новых: 49 | По запросу: 98 | Всего: 98

  📄 Страница 3 / 5
    📦 Карточек: 50
  ✅ Новых: 50 | По запросу: 148 | Всего: 148

  📄 Страница 4 / 5
    📦 Карточек: 49
  ✅ Новых: 49 | По запросу: 197 | Всего: 197

  📄 Страница 5 / 5
    📦 Карточек: 49
  ✅ Новых: 49 | По запросу: 246 | Всего: 246

═══════════════════════════════════════════════════════
🔍 [2/69] «специалист по вводу данных»
═══════════════════════════════════════════════════════
  📑 Страниц найдено: 3

  📄 Страница 1 / 3
    📦 Карточек: 49
  ✅ Новых: 49 | По запросу: 49 | Всего: 295

  📄 Страница 2 / 3
    📦 Карточек: 50
  ✅ Новых: 50 | По запросу: 99 | Всего: 345

  📄 Страница 3 / 3
    📦 Карточек: 31
  ✅ Новых: 31 | По запросу: 130 |

In [5]:
df.to_excel('df.xlsx')

In [5]:
'''обработка Excel'''
import pandas as pd
from openpyxl import load_workbook
from openpyxl.styles import Alignment
from openpyxl.utils import get_column_letter
from openpyxl.worksheet.datavalidation import DataValidation
import time

INPUT_FILE = "hh_vacancies.xlsx"
OUTPUT_FILE = "hh_vacancies_p.xlsx"

COL_STATUS = "статус"
COL_TITLE  = "название"
COL_LINK   = "ссылка"
COL_QUERY  = "запрос"

STATUS_VALUES = "Мусор,Следующее,Требования,Сложно,Отправлено,Next!!!,Повтор"

TITLE_COL_WIDTH = 40
LINK_COL_WIDTH  = 5.5
MIN_COL_WIDTH   = 8
MAX_QUERY_WIDTH = 60

def log(msg):
    print(f"[{time.strftime('%H:%M:%S')}] {msg}", flush=True)

# ШАГ 1: Читаем только непустые столбцы
log("📂 Читаем файл...")
raw = pd.read_excel(INPUT_FILE, header=0)

raw = raw.loc[:, ~raw.columns.str.startswith("Unnamed")]
raw = raw.dropna(axis=1, how="all")
raw = raw.dropna(axis=0, how="all")

log(f"✅ Прочитано: {len(raw)} строк, {len(raw.columns)} столбцов")
log(f"   Столбцы: {list(raw.columns)}")

# ШАГ 2: Проверяем нужные столбцы
log("🔍 Проверяем нужные столбцы...")
missing = [c for c in [COL_TITLE, COL_LINK, COL_QUERY] if c not in raw.columns]
if missing:
    raise ValueError(f"Не найдены столбцы: {missing}\nДоступные: {list(raw.columns)}")
log("✅ Все столбцы найдены")

# ШАГ 3: Переставляем «запрос» на 4-е место
log(f"🔀 Переставляем «{COL_QUERY}» на 4-ю позицию...")
cols = list(raw.columns)
cols.remove(COL_QUERY)
cols.insert(3, COL_QUERY)
df = raw[cols]
log(f"✅ Порядок: {list(df.columns)}")

# ШАГ 4: Сохраняем чистые данные
log(f"💾 Записываем в {OUTPUT_FILE}...")
df.to_excel(OUTPUT_FILE, index=False)
log("✅ Данные записаны")

# ШАГ 5: Открываем через openpyxl для форматирования
log("🔧 Открываем для форматирования...")
wb = load_workbook(OUTPUT_FILE)
ws = wb.active
log(f"   Строк: {ws.max_row}, столбцов: {ws.max_column}")

header = [ws.cell(1, c).value for c in range(1, ws.max_column + 1)]
col_idx = {name: i + 1 for i, name in enumerate(header)}

title_col  = col_idx[COL_TITLE]
link_col   = col_idx[COL_LINK]
query_col  = col_idx[COL_QUERY]
status_col = col_idx.get(COL_STATUS)

# ШАГ 6: Автофильтр
log("🔽 Ставим автофильтр...")
ws.auto_filter.ref = ws.dimensions
log("✅ Автофильтр установлен")

# ШАГ 7: Выпадающий список для столбца «статус»
if status_col:
    status_letter = get_column_letter(status_col)
    log(f"📋 Добавляем выпадающий список для «{COL_STATUS}» (столбец {status_letter})...")

    dv = DataValidation(
        type="list",
        formula1=f'"{STATUS_VALUES}"',
        allow_blank=True,
        showDropDown=False,   # False = стрелочка видна в Excel
        showErrorMessage=False,
    )
    ws.add_data_validation(dv)

    # Применяем ко всем строкам данных, значения существующих ячеек НЕ трогаем
    data_range = f"{status_letter}2:{status_letter}{ws.max_row}"
    dv.sqref = data_range
    log(f"✅ Выпадающий список добавлен на диапазон {data_range}")
else:
    log(f"⚠️  Столбец «{COL_STATUS}» не найден — выпадающий список не добавлен")

# ШАГ 8: Форматируем столбцы
log(f"🎨 Форматируем {ws.max_column} столбцов...")

for col_num in range(1, ws.max_column + 1):
    col_letter = get_column_letter(col_num)
    col_name = header[col_num - 1]

    if col_num == title_col:
        log(f"   📝 [{col_letter}] «{col_name}» → перенос строк, ширина {TITLE_COL_WIDTH}")
        ws.column_dimensions[col_letter].width = TITLE_COL_WIDTH
        for row in range(2, ws.max_row + 1):
            ws.cell(row, col_num).alignment = Alignment(wrap_text=True, vertical="top")

    elif col_num == link_col:
        log(f"   🔗 [{col_letter}] «{col_name}» → ширина {LINK_COL_WIDTH} (~1 см)")
        ws.column_dimensions[col_letter].width = LINK_COL_WIDTH
        for row in range(2, ws.max_row + 1):
            ws.cell(row, col_num).alignment = Alignment(wrap_text=False, vertical="top")

    elif col_num == query_col:
        max_len = max(
            (len(str(ws.cell(row, col_num).value))
             for row in range(1, ws.max_row + 1)
             if ws.cell(row, col_num).value),
            default=10
        )
        width = min(max_len + 2, MAX_QUERY_WIDTH)
        log(f"   🔎 [{col_letter}] «{col_name}» → авто-ширина {width}")
        ws.column_dimensions[col_letter].width = width
        for row in range(2, ws.max_row + 1):
            ws.cell(row, col_num).alignment = Alignment(wrap_text=False, vertical="top")

    else:
        log(f"   📏 [{col_letter}] «{col_name}» → минимальная ширина {MIN_COL_WIDTH}")
        ws.column_dimensions[col_letter].width = MIN_COL_WIDTH
        for row in range(2, ws.max_row + 1):
            ws.cell(row, col_num).alignment = Alignment(wrap_text=False, vertical="top")

# ШАГ 9: Заголовки
log("🏷️  Форматируем заголовки...")
for col_num in range(1, ws.max_column + 1):
    ws.cell(1, col_num).alignment = Alignment(horizontal="center", vertical="center", wrap_text=False)

# ШАГ 10: Финальное сохранение
log("💾 Сохраняем итоговый файл...")
wb.save(OUTPUT_FILE)
log(f"🎉 Готово! Файл: {OUTPUT_FILE}")

[11:42:12] 📂 Читаем файл...
[11:42:12] ✅ Прочитано: 343 строк, 10 столбцов
[11:42:12]    Столбцы: ['статус', 'дата', 'название', 'запрос', 'зарплата', 'опыт', 'город', 'ссылка', 'компания', 'все_тексты']
[11:42:12] 🔍 Проверяем нужные столбцы...
[11:42:12] ✅ Все столбцы найдены
[11:42:12] 🔀 Переставляем «запрос» на 4-ю позицию...
[11:42:12] ✅ Порядок: ['статус', 'дата', 'название', 'запрос', 'зарплата', 'опыт', 'город', 'ссылка', 'компания', 'все_тексты']
[11:42:12] 💾 Записываем в hh_vacancies_p.xlsx...
[11:42:12] ✅ Данные записаны
[11:42:12] 🔧 Открываем для форматирования...
[11:42:12]    Строк: 344, столбцов: 10
[11:42:12] 🔽 Ставим автофильтр...
[11:42:12] ✅ Автофильтр установлен
[11:42:12] 📋 Добавляем выпадающий список для «статус» (столбец A)...
[11:42:12] ✅ Выпадающий список добавлен на диапазон A2:A344
[11:42:12] 🎨 Форматируем 10 столбцов...
[11:42:12]    📏 [A] «статус» → минимальная ширина 8
[11:42:12]    📏 [B] «дата» → минимальная ширина 8
[11:42:12]    📝 [C] «название» → перено

In [6]:
Данные и операторы
"Оператор по вводу данных", "Оператор ПК", "Специалист по обработке данных", "Оператор базы данных", "Специалист по оцифровке данных"
Модерация
"Модератор контента", "Модератор", "Специалист по модерации", "Верификатор данных", "Асессор", "Разметчик данных"
Поддержка и чат
"Специалист службы поддержки", "Оператор чата", "Менеджер по работе с клиентами", "Менеджер переписки", "Специалист технической поддержки"
Мессенджеры и рассылки
"Менеджер по рассылкам", "Менеджер Telegram", "Администратор Telegram-канала", "Специалист по email-рассылкам", "Менеджер мессенджеров"
SMM и контент
"Контент-менеджер", "SMM-менеджер", "Администратор социальных сетей", "Менеджер социальных сетей", "Редактор контента"
Ассистент и администратор
"Помощник руководителя", "Личный ассистент", "Удаленный ассистент", "Онлайн-ассистент", "Офис-менеджер", "Административный ассистент"
Маркетплейсы
"Менеджер маркетплейса", "Менеджер Wildberries", "Менеджер Ozon", "Специалист по маркетплейсам", "Оператор карточек товаров"
Аналитика и исследования
"Аналитик данных", "Интернет-исследователь", "Ресёрчер", "Специалист по мониторингу", "Сборщик информации"
Тексты и переводы
"Копирайтер", "Рерайтер", "Транскрибатор", "Переводчик", "Технический писатель", "Редактор"
HR и рекрутинг
"HR-ассистент", "Рекрутер", "Менеджер по подбору персонала"
Продажи и лиды
"Менеджер по продажам", "Менеджер по холодным рассылкам", "Лидогенератор", "Менеджер по развитию"
Тестирование
"Тестировщик", "QA-инженер", "Специалист по тестированию"

In [3]:

# Пути к файлам и листам (замените на свои)
file1 = 'hh_vacancies.xlsx'
sheet1 = 'Вакансии'  # или номер, напр. 0
cols1 = ['статус', 'дата', 'название', 'запрос', 'зарплата', 'опыт', 'город', 'ссылка']  # столбцы для первой таблицы

file2 = 'hh_vacancies_mn.xlsx'
sheet2 = 'Вакансии'
cols2 = ['статус', 'дата', 'название', 'запрос', 'зарплата', 'опыт', 'город', 'ссылка']  # столбцы для второй таблицы

# Чтение таблиц с выбором столбцов
df1 = pd.read_excel(file1, sheet_name=sheet1, usecols=cols1)[cols1]
df2 = pd.read_excel(file2, sheet_name=sheet2, usecols=cols2)[cols2]

# Объединение: merge по ключу 'ID' (inner join) или замените на pd.concat([df1, df2], axis=0, ignore_index=True) для склейки по строкам
result = pd.merge(df1, df2, on=['название', 'зарплата', 'опыт', 'город'], how='outer')  # варианты: 'left', 'right', 'outer'
# result = pd.merge(df1, df2, on=['ID', 'Дата'], how='outer')

# Удаление дублей по ключам (оставит первое вхождение)
result = result.drop_duplicates(subset=['название', 'зарплата', 'опыт', 'город'])

# Сохранение в новый файл
result.to_excel('объединенный_результат.xlsx', index=False)
print('Готово! Файл сохранен: объединенный_результат.xlsx')
# print(result.head())  # просмотр первых строк

Готово! Файл сохранен: объединенный_результат.xlsx


In [16]:
import time
import pandas as pd
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.common.exceptions import TimeoutException, NoSuchElementException
import openpyxl

# ===================== НАСТРОЙКИ =====================
EXCEL_FILE = "vacancies.xlsx"  # Путь к Excel файлу
EMAIL = "anfisso@yahoo.com"

COVER_LETTER = """Добрый день!

Опыт работы с данными — более 4 лет, работала с проектами разного масштаба.

Стек:
— Python — предобработка и очистка данных, EDA, машинное обучение, статистические и бизнес-расчёты
— Excel, Google Таблицы — отчётность, сводные таблицы, автоматизация
— SQL — написание запросов, выгрузка и анализ данных
— BI-инструменты: Tableau, DataLens, Metabase — построение дашбордов и визуализация метрик

Удалённый формат работы

https://t.me/anfisso"""

# ===================== ФИЛЬТРАЦИЯ =====================
# Формат:
# {
#   "номер_столбца (0-based)": {
#       "mode": "exact" | "contains" | "empty_or_values",
#       "values": ["значение1", "значение2"]
#   }
# }
FILTERS = {
    0: {
        "mode": "empty_or_values",       # пустое ИЛИ одно из values
        "values": ["мусор"]
    },
        3: {
        "mode": "contains",         # пустое ИЛИ одно из values
        "values": ["аналитик"]
    },
    # Пример второго фильтра:
    # 3: {
    #     "mode": "contains",            # ячейка содержит хотя бы одно из values
    #     "values": ["Москва", "Удалённо"]
    # },
}

LINK_COLUMN = "ссылка"   # Название столбца со ссылками (заголовок)
SHEET_NAME  = "Вакансии" # Название листа
# =====================================================


def match_filter(cell_value, rule):
    val = str(cell_value).strip() if cell_value is not None else ""
    mode = rule["mode"]
    values = [str(v).strip() for v in rule["values"]]

    if mode == "exact":
        return val in values
    elif mode == "contains":
        return any(v.lower() in val.lower() for v in values)
    elif mode == "empty_or_values":
        return val == "" or val.lower() in [v.lower() for v in values]
    return False


def load_and_filter_excel(filepath):
    wb = openpyxl.load_workbook(filepath)
    ws = wb[SHEET_NAME]

    headers = [cell.value for cell in ws[1]]
    link_col_idx = headers.index(LINK_COLUMN)  # 0-based

    rows_to_process = []
    for row_idx, row in enumerate(ws.iter_rows(min_row=2, values_only=False), start=2):
        values = [cell.value for cell in row]

        # Проверяем все фильтры
        passed = True
        for col_idx, rule in FILTERS.items():
            cell_val = values[col_idx] if col_idx < len(values) else None
            if not match_filter(cell_val, rule):
                passed = False
                break

        if passed:
            link = values[link_col_idx]
            if link:
                rows_to_process.append((row_idx, link))

    return wb, ws, rows_to_process, link_col_idx


def update_cell(ws, row_idx, col_idx, value, filepath, wb):
    ws.cell(row=row_idx, column=col_idx + 1).value = value  # openpyxl — 1-based
    wb.save(filepath)


def login_hh(driver):
    wait = WebDriverWait(driver, 15)
    driver.get("https://hh.ru")
    time.sleep(2)

    # Кнопка "Войти" в шапке
    btn = wait.until(EC.element_to_be_clickable(
        (By.XPATH, "/html/body/div[5]/div/div[3]/div/div/div/div[8]/a/span/span/span")))
    btn.click()
    time.sleep(1)

    # Кнопка "Войти" в модалке
    btn2 = wait.until(EC.element_to_be_clickable(
        (By.XPATH, "/html/body/div[2]/div/div/div[1]/div/div/div[1]/div/div/div/div/div/div[1]/div/div/form/div/div[2]/div[2]/div/button[1]/span/span/span")))
    btn2.click()
    time.sleep(1)

    # Выбор ввода почты
    email_tab = wait.until(EC.element_to_be_clickable(
        (By.XPATH, "/html/body/div[2]/div/div/div[1]/div/div/div[1]/div/div/div/div/div/div[1]/div/div/form/div/div/div[2]/div/div[1]/label[2]/div")))
    email_tab.click()
    time.sleep(1)

    # Ввод почты
    email_input = wait.until(EC.presence_of_element_located(
        (By.XPATH, "/html/body/div[2]/div/div/div[1]/div/div/div[1]/div/div/div/div/div/div[1]/div/div/form/div/div/div[3]/div[1]/div/div[2]/div[2]/input")))
    email_input.clear()
    email_input.send_keys(EMAIL)

    # Подтверждение отправки кода
    submit = driver.find_element(By.XPATH,
        "//button[@type='submit']")
    submit.click()

    print("\n⏳ Введите код из письма на почте в браузере. Ожидание 90 секунд...")
    time.sleep(60)
    print("✅ Продолжаем работу.")

def process_vacancy(driver, url, row_idx, ws, wb, col_idx):
    wait = WebDriverWait(driver, 10)
    driver.get(url)
    time.sleep(2)

    # Ищем все кнопки/ссылки со словом "откликнуться" и кликаем первую
    try:
        # Ищем по тексту — охватывает button, a, span внутри них
        apply_buttons = driver.find_elements(
            By.XPATH,
            "//*[contains(translate(normalize-space(text()), "
            "'АБВГДЕЁЖЗИЙКЛМНОПРСТУФХЦЧШЩЪЫЬЭЮЯ', "
            "'абвгдеёжзийклмнопрстуфхцчшщъыьэюя'), 'откликнуться')]"
        )

        if not apply_buttons:
            raise TimeoutException("Кнопка не найдена")

        # Берём первую видимую кнопку
        clicked = False
        for btn in apply_buttons:
            try:
                if btn.is_displayed() and btn.is_enabled():
                    driver.execute_script("arguments[0].click();", btn)
                    clicked = True
                    break
            except Exception:
                continue

        if not clicked:
            raise TimeoutException("Ни одна кнопка не кликабельна")

        time.sleep(3)  # Пауза чтобы выбрать резюме

    except TimeoutException:
        print(f"  ⚠️  Кнопка откликнуться не найдена: {url}")
        ws.cell(row=row_idx, column=1).value = "Stop"
        wb.save(EXCEL_FILE)
        return

    # Ищем textarea для сопроводительного письма
    try:
        textarea = wait.until(EC.presence_of_element_located(
            (By.XPATH, "/html/body/div[28]/div/div[1]/div[3]/div[1]/div[1]/div/div/form/div/div[2]/div[1]/div/div[1]/textarea")))
        textarea.clear()
        textarea.send_keys(COVER_LETTER)
        time.sleep(1)

        # Кнопка "Отправить"
        send_btn = wait.until(EC.element_to_be_clickable(
            (By.XPATH, "/html/body/div[28]/div/div[3]/div/div/div/div/div")))
        send_btn.click()
        time.sleep(2)

        ws.cell(row=row_idx, column=1).value = "Send-hh"
        wb.save(EXCEL_FILE)
        print(f"  ✅ Отклик отправлен: {url}")

    except TimeoutException:
        print(f"  ⚠️  Textarea не найдена (тест/нестандартная форма): {url}")
        ws.cell(row=row_idx, column=1).value = "Stop"
        wb.save(EXCEL_FILE)

def main():
    print("📂 Загрузка Excel...")
    wb, ws, rows, link_col_idx = load_and_filter_excel(EXCEL_FILE)
    print(f"   Найдено вакансий для обработки: {len(rows)}")

    options = webdriver.ChromeOptions()
    options.add_argument("--start-maximized")
    # options.add_argument("--headless")  # раскомментировать для фонового режима
    driver = webdriver.Chrome(options=options)

    try:
        print("\n🔐 Авторизация на hh.ru...")
        login_hh(driver)

        for i, (row_idx, url) in enumerate(rows, 1):
            print(f"\n[{i}/{len(rows)}] Обработка: {url}")
            try:
                process_vacancy(driver, url, row_idx, ws, wb, link_col_idx)
            except Exception as e:
                print(f"  ❌ Ошибка: {e}")
                ws.cell(row=row_idx, column=1).value = "Stop"
                wb.save(EXCEL_FILE)
            time.sleep(2)

    finally:
        driver.quit()
        print("\n🏁 Скрипт завершён.")


if __name__ == "__main__":
    main()

📂 Загрузка Excel...
   Найдено вакансий для обработки: 34

🔐 Авторизация на hh.ru...

⏳ Введите код из письма на почте в браузере. Ожидание 90 секунд...
✅ Продолжаем работу.

[1/34] Обработка: https://hh.ru/vacancy/131616709?query=%D0%B0%D0%BD%D0%B0%D0%BB%D0%B8%D1%82%D0%B8%D0%BA+%D0%B4%D0%B0%D0%BD%D0%BD%D1%8B%D1%85&hhtmFrom=vacancy_search_list
  ⚠️  Textarea не найдена (тест/нестандартная форма): https://hh.ru/vacancy/131616709?query=%D0%B0%D0%BD%D0%B0%D0%BB%D0%B8%D1%82%D0%B8%D0%BA+%D0%B4%D0%B0%D0%BD%D0%BD%D1%8B%D1%85&hhtmFrom=vacancy_search_list

[2/34] Обработка: https://hh.ru/vacancy/131471492?query=%D0%B0%D0%BD%D0%B0%D0%BB%D0%B8%D1%82%D0%B8%D0%BA+%D0%B4%D0%B0%D0%BD%D0%BD%D1%8B%D1%85&hhtmFrom=vacancy_search_list
  ⚠️  Textarea не найдена (тест/нестандартная форма): https://hh.ru/vacancy/131471492?query=%D0%B0%D0%BD%D0%B0%D0%BB%D0%B8%D1%82%D0%B8%D0%BA+%D0%B4%D0%B0%D0%BD%D0%BD%D1%8B%D1%85&hhtmFrom=vacancy_search_list

[3/34] Обработка: https://hh.ru/vacancy/131430661?query=%D0%B0%D0%B

KeyboardInterrupt: 